# 1 — Data Gathering

Notebook ini memuat data panel mentah dan menyimpannya sebagai CSV antar-tahap (pipeline berbasis file).

**Input (project root):** `model_panel.csv`

**Output:** `/output/1_raw_panel_data.csv`

Catatan:
1. Baris dengan `provinsi_id == 19` dihapus dari dataset sesuai kebutuhan analisis.
2. Data `x5_penetrasi_internet_pct` untuk tahun 2021 diimputasi menggunakan *backward linear extrapolation* agar tren kenaikan historis tetap terjaga.

In [ ]:
from pathlib import Path
import pandas as pd

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / 'model_panel.csv').exists():
            return p
    raise FileNotFoundError('Could not find model_panel.csv in current or parent directories.')

ROOT = find_project_root(Path.cwd())
input_path = ROOT / 'model_panel.csv'
output_dir = ROOT / 'pipeline_A' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'A0_raw_panel_data.csv'

df = pd.read_csv(input_path)

initial_rows = len(df)
df = df[df['provinsi_id'] != 19].copy()
removed_rows = initial_rows - len(df)

print(f'Loaded: {input_path}')
print(f'Removed {removed_rows:,} rows where provinsi_id == 19.')
print(f'Shape before imputation: {df.shape[0]:,} rows × {df.shape[1]} columns')

# IMPUTASI x5_penetrasi_internet_pct UNTUK TAHUN 2021
# Menggunakan backward linear extrapolation: x5_2021 = 2 * x5_2022 - x5_2023
x5_yearly = df.groupby(['provinsi_id', 'tahun'])['x5_penetrasi_internet_pct'].first().unstack('tahun')
x5_2021_imputed = 2 * x5_yearly[2022] - x5_yearly[2023]
x5_2021_imputed = x5_2021_imputed.clip(lower=0) # Mencegah nilai persentase negatif

for pid in x5_2021_imputed.index:
    df.loc[(df['provinsi_id'] == pid) & (df['tahun'] == 2021), 'x5_penetrasi_internet_pct'] = x5_2021_imputed[pid]

print('\nContoh hasil imputasi 2021 vs 2022 vs 2023 (DKI Jakarta):')
dki = df[df['provinsi_id'] == 2].groupby('tahun')['x5_penetrasi_internet_pct'].first()
print(dki.loc[[2021, 2022, 2023]])

display(df.head())

In [ ]:
# Basic validation + persist artifact
assert (df['provinsi_id'] == 19).sum() == 0, 'provinsi_id 19 still present after filtering.'
assert df['x5_penetrasi_internet_pct'].isna().sum() < len(df), 'Imputation might have failed, still many NaNs.'

df.to_csv(output_path, index=False)
print(f'Saved: {output_path}')